# Camera Manual RAG System

This portfolio notebook keeps the key project results from the original Colab assignment while omitting long installation logs, download widgets, raw vector database artifacts, and private API keys.

## Project Goal

Camera manuals are long, technical, and difficult to search manually. This project built a RAG-based assistant that retrieves relevant manual sections and generates grounded answers to camera operation questions.

## Methods

- PDF parsing with `unstructured` high-resolution partitioning.
- OCR with `pytesseract` for image-based manual content.
- Semantic sentence-based chunking with metadata for model and page traceability.
- Dense embeddings using `intfloat/e5-base-v2`.
- ChromaDB vector storage.
- Hybrid retrieval combining BM25 and vector search.
- HyDE-style hypothetical answer embeddings for retrieval enhancement.
- Cohere reranking to improve context ordering.
- Gemini 2.0 Flash for final answer generation.
- RAGAS evaluation for faithfulness, answer relevancy, and context precision.

In [ ]:
test_questions = [
    'How do I enable silent shooting on the OM-1?',
    'What can I do for using Super Control Panel the OM-1?',
    'How do I turn on face priority AF on the OM-1?',
    'How do I transfer images to a smartphone on the E-M1 Mark II?',
    'What types of cards can be used with E-M1 Mark II?',
    'List all the lenses that can be used with the E-M1 Mark II?',
    'Where do I find the ISO sensitivity settings in the E-M5 Mark II?',
    'What should I be cautious about if I want to edit photos with the E-M5 Mark II?',
    'How can I format the memory card on the E-M10 Mark II?',
    'How to connect Wi-Fi on the E-M10 Mark II?',
    'Where and how can I set the self timer on the E-M10 Mark II?',
    'When shooting in A mode, what does it mean if the shutter speed display is blinking?',
    'What is P mode? How does it differ from A/S/M modes?'
]

## Test Questions and Results

The original notebook tested 13 camera-operation questions across OM-1, E-M1 Mark II, E-M5 Mark II, and E-M10 Mark II manuals. The table below consolidates the original generated answers into a portfolio-readable format while preserving the full set of test cases.

| # | Test Question | Generated Answer / Result Summary | Retrieved Context Count |
|---:|---|---|---:|
| 1 | How do I enable silent shooting on the OM-1? | The answer points users to the OM-1 manual section on shooting without shutter sound, specifically Silent Settings around page 132, and notes related flash-mode configuration around page 127. | 5 |
| 2 | What can I do for using Super Control Panel the OM-1? | The answer explains that the Super Control Panel / LV Super Control Panel lists current shooting settings and values, with different usage depending on whether the user frames through the viewfinder or monitor. | 18 |
| 3 | How do I turn on face priority AF on the OM-1? | The answer explains that the manual may not use the exact phrase Face Priority AF, but describes face/eye detection behavior, AF operation settings, and face selection via assigned buttons and dials. It also notes that the answer needed clarification because the feature is described indirectly. | 12 |
| 4 | How do I transfer images to a smartphone on the E-M1 Mark II? | The answer gives a concise workflow: connect the camera to a smartphone, then launch OI.Share and use the Image Transfer button, with references to the manual pages around 135-136. | 9 |
| 5 | What types of cards can be used with the E-M1 Mark II? | The answer identifies compatible card types as SD, SDHC, SDXC, and Eye-Fi cards. | 6 |
| 6 | List all the lenses that can be used with the E-M1 Mark II. | The answer groups compatible lenses into Micro Four Thirds lenses, Four Thirds lenses with an adapter, OM System lenses with a mount adapter, and lenses supporting Pro Capture shooting, with a note to check Olympus resources for compatibility details. | 6 |
| 7 | Where do I find the ISO sensitivity settings in the E-M5 Mark II? | The generated answer became broader than the retrieved context. It explains likely access paths through the camera menu, Super Control Panel, and custom buttons, while also discussing ISO range, Auto ISO, image stabilization, noise, and exposure trade-offs. | 1 |
| 8 | What should I be cautious about if I want to edit photos with the E-M5 Mark II? | The answer highlights RAW Data Edit cautions: adjust camera settings before choosing RAW Data Edit Current, and note that JPEG copies are processed using the current camera settings. | 13 |
| 9 | How can I format the memory card on the E-M10 Mark II? | The answer gives a corrected step-by-step path: press Menu, open the Setup menu, choose Format, select Yes, and confirm with OK. It also warns that formatting erases all card data. | 3 |
| 10 | How to connect Wi-Fi on the E-M10 Mark II? | The answer says to select Wi-Fi Connect Settings and, if the QR code cannot be read, manually enter the SSID and password in the smartphone Wi-Fi settings. | 3 |
| 11 | Where and how can I set the self timer on the E-M10 Mark II? | The answer provides a fuller workflow: open the menu, go to Shooting Menu 2, choose Drive Mode, select the desired self-timer option, confirm, optionally customize self-timer behavior, then half-press to focus and fully press to start. | 9 |
| 12 | When shooting in A mode, what does it mean if the shutter speed display is blinking? How can this be resolved? | The answer explains that blinking shutter speed in Aperture Priority usually means the camera cannot find a proper shutter speed for the selected aperture and ISO. It suggests adjusting aperture, ISO, exposure compensation, lighting, or using an ND filter if the scene is too bright. | 17 |
| 13 | What is P mode? How does it differ from A/S/M modes, and when is it suitable? | The answer explains Program AE mode, contrasts it with Aperture Priority, Shutter Priority, and Manual modes, describes Program Shift, and recommends P mode for quick shooting, general photography, changing light, beginner use, and fast-moving scenes. | 14 |


## Result Observations

- The system performed well on concrete operation questions where the manual contained direct procedural evidence, such as silent shooting, smartphone transfer, card formatting, Wi-Fi setup, and self-timer configuration.
- Broader conceptual questions, such as P mode or blinking shutter speed in A mode, produced more explanatory answers and required stronger synthesis across retrieved contexts.
- Some answers showed the need for stricter grounding. For example, the ISO sensitivity answer expanded beyond the retrieved context and was flagged in the original analysis as needing more careful support from the manual.
- The results show why retrieval quality, reranking, and answer verification are important in technical-manual RAG systems.

## RAGAS Evaluation Results

The system was evaluated on 13 camera-operation questions.

| Metric | Score | Interpretation |
|---|---:|---|
| Context Precision Without Reference | 0.8510 | Strong retrieval of relevant manual passages. |
| Answer Relevancy | 0.8499 | Generated answers were highly relevant to user questions. |
| Faithfulness | 0.7614 | Most answers were grounded in retrieved context, with room for improvement. |

In [ ]:
ragas_results = {
    'faithfulness': 0.7614,
    'answer_relevancy': 0.8499,
    'llm_context_precision_without_reference': 0.8510,
}
print('=== RAGAS Evaluation Results ===')
print(ragas_results)

=== RAGAS Evaluation Results ===
{'faithfulness': 0.7614, 'answer_relevancy': 0.8499, 'llm_context_precision_without_reference': 0.8510}


## Limitations

The main limitation was visual-symbol understanding. Camera manuals contain icons and UI symbols that can be difficult for a text-focused RAG system to interpret. Repeated manual content across modes also introduced retrieval noise. A future multimodal RAG system could improve diagram and icon interpretation.